# 04 Serve Recommendation

Build a reliability-aware recommendation table by context.

In [ ]:
from pathlib import Path
import pandas as pd

from src.recommend_serves import wilson_ci, reliability_label, compute_recommendation_score


In [ ]:
df = pd.read_csv(Path('../data/processed/serves_cleaned.csv'))
combo_cols = ['serve_type','spin_type','serve_length','placement_zone']
group_cols = ['opponent_style','game_state'] + combo_cols

agg = (
    df.groupby(group_cols, dropna=False)
      .agg(attempts=('point_win','size'), wins=('point_win','sum'))
      .reset_index()
)
agg['pred_win_prob'] = agg['wins'] / agg['attempts']
agg['pred_weak_return_prob'] = agg['pred_win_prob']  # placeholder until dedicated model
agg['pred_intended_rally_prob'] = agg['pred_win_prob']  # placeholder until dedicated model
agg['reliability_score'] = (agg['attempts'].clip(upper=40) / 40.0)
agg['reliability'] = agg['attempts'].apply(reliability_label)


In [ ]:
cis = agg.apply(lambda r: wilson_ci(int(r['wins']), int(r['attempts'])), axis=1)
agg['ci_low'] = [c[0] for c in cis]
agg['ci_high'] = [c[1] for c in cis]
ranked = compute_recommendation_score(agg)
ranked.head(15)

In [ ]:
recommendations = (
    ranked.sort_values('serve_score', ascending=False)
          .groupby(['opponent_style','game_state'], dropna=False)
          .head(1)
          [['opponent_style','game_state','serve_type','spin_type','serve_length','placement_zone','serve_score','attempts','ci_low','ci_high','reliability']]
)
recommendations.sort_values(['opponent_style','game_state'])

## Interpretation guardrails
- Recommendations are associative, not causal.
- Low-attempt combinations should be treated as exploratory.
- Replace placeholder probabilities with dedicated weak-return and intended-rally models when available.
